In [2]:
# [Cell 1]: Environment Setup and Data Loading
import os
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

DATA_DIR = Path("../dataset")
MEM_DIR = Path(".")

# Load cleaned interactions
train_df = pd.read_csv(DATA_DIR / "train_cleaned.csv")
test_df = pd.read_csv(DATA_DIR / "test_cleaned.csv")

NUM_USERS = int(train_df["user_idx"].max() + 1)
NUM_GAMES = int(train_df["game_idx"].max() + 1)

print(f"Users in Training: {NUM_USERS:,} | Games in Training: {NUM_GAMES:,}")
print(f"Train Interactions: {len(train_df):,} | Test Interactions: {len(test_df):,}")

Users in Training: 329,719 | Games in Training: 83,630
Train Interactions: 1,045,048 | Test Interactions: 316,273


In [3]:
# [Cell 2]: Build User-Item Interaction Matrix
rows = train_df["user_idx"].values
cols = train_df["game_idx"].values
ratings = train_df["rating"].values

# Raw interaction sparse matrix
user_item_matrix = csr_matrix((ratings, (rows, cols)), shape=(NUM_USERS, NUM_GAMES))
save_npz(MEM_DIR / "user_item_matrix.npz", user_item_matrix)
print("Sparse User-Item Matrix shape:", user_item_matrix.shape)
print(f"Matrix Non-zeros: {user_item_matrix.nnz:,}")

Sparse User-Item Matrix shape: (329719, 83630)
Matrix Non-zeros: 1,045,048


In [4]:
# [Cell 3]: Experiment 1 - Uncentered Item-Item Cosine Similarity
print("Running Experiment 1: Uncentered Cosine Similarity...")
item_user_matrix = user_item_matrix.T
cosine_sim_matrix = cosine_similarity(item_user_matrix, dense_output=False)
print("Standard Cosine Similarity Matrix ready.")

Running Experiment 1: Uncentered Cosine Similarity...
Standard Cosine Similarity Matrix ready.


In [5]:
# [Cell 4]: Experiment 2 - User Mean-Centering (Adjusted Cosine)
print("Running Experiment 2: Computing User-Centered Cosine Matrix...")
user_sums = np.array(user_item_matrix.sum(axis=1)).flatten()
user_counts = np.diff(user_item_matrix.indptr)
user_means = np.zeros(NUM_USERS)
nonzero_mask = user_counts > 0
user_means[nonzero_mask] = user_sums[nonzero_mask] / user_counts[nonzero_mask]

# Center the ratings
centered_ratings = ratings - user_means[rows]
user_item_centered = csr_matrix((centered_ratings, (rows, cols)), shape=(NUM_USERS, NUM_GAMES))

# Pearson proxy similarity
pearson_sim_matrix = cosine_similarity(user_item_centered.T, dense_output=False)
print("Centered Cosine (Pearson Proxy) Matrix ready.")

Running Experiment 2: Computing User-Centered Cosine Matrix...
Centered Cosine (Pearson Proxy) Matrix ready.


In [6]:
# [Cell 5]: Experiment 3 - Top-50 Neighbor Sparsification
def prune_top_k(sim_matrix, top_n=50):
    print(f"Running Experiment 3: Pruning similarity matrix to Top-{top_n} neighbors...")
    sim_coo = sim_matrix.tocoo()
    df_sim = pd.DataFrame({'row': sim_coo.row, 'col': sim_coo.col, 'data': sim_coo.data})
    
    # Exclude self-similarity
    df_sim = df_sim[df_sim['row'] != df_sim['col']]
    
    # Retain top_n similar items per item
    df_sim = df_sim.sort_values(['row', 'data'], ascending=[True, False])
    df_top = df_sim.groupby('row').head(top_n)
    
    return csr_matrix((df_top['data'], (df_top['row'], df_top['col'])), shape=sim_matrix.shape)

pruned_cosine_matrix = prune_top_k(cosine_sim_matrix, top_n=50)
print("Pruned Similarity Matrix built.")

Running Experiment 3: Pruning similarity matrix to Top-50 neighbors...
Pruned Similarity Matrix built.


In [7]:
# [Cell 6]: Evaluating All Experiments on Test Set (K=10)
K = 10
discounts = 1 / np.log2(np.arange(2, K + 2))
train_history = train_df.groupby("user_idx")["game_idx"].apply(set).to_dict()
rel_by_user = test_df[test_df["rating"] >= 4.0].groupby("user_idx")["game_idx"].apply(set).to_dict()

rng = np.random.default_rng(42)
sampled_users = rng.choice(list(rel_by_user.keys()), size=min(500, len(rel_by_user)), replace=False)

def evaluate_sim_matrix(sim_mat, name="Model"):
    precisions, recalls, ndcgs = [], [], []
    for u in tqdm(sampled_users, desc=f"Evaluating {name}"):
        u_ratings = user_item_matrix[int(u)].toarray().flatten()
        scores = sim_mat.dot(u_ratings)

        seen = train_history.get(int(u), set())
        if seen:
            scores[list(seen)] = -np.inf

        top_items = np.argpartition(scores, -K)[-K:]
        top_items = top_items[np.argsort(scores[top_items])[::-1]]

        rel = rel_by_user[int(u)]
        hits = np.array([1 if g in rel else 0 for g in top_items])

        precisions.append(hits.sum() / K)
        recalls.append(hits.sum() / len(rel) if rel else 0.0)
        dcg = np.sum(hits * discounts)
        idcg = discounts[:min(len(rel), K)].sum()
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

    return {
        "Experiment": name,
        "Precision@10": round(float(np.mean(precisions)), 4),
        "Recall@10": round(float(np.mean(recalls)), 4),
        "NDCG@10": round(float(np.mean(ndcgs)), 4)
    }

res1 = evaluate_sim_matrix(cosine_sim_matrix, "Exp 1: Standard Cosine")
res2 = evaluate_sim_matrix(pearson_sim_matrix, "Exp 2: User-Centered Cosine")
res3 = evaluate_sim_matrix(pruned_cosine_matrix, "Exp 3: Top-50 Pruned Cosine")

memory_experiments_df = pd.DataFrame([res1, res2, res3])
print("\n" + "="*50)
print(memory_experiments_df.to_string(index=False))
memory_experiments_df.to_csv(MEM_DIR / "memory_experiments_ablation.csv", index=False)

Evaluating Exp 3: Top-50 Pruned Cosine: 100%|██████████| 500/500 [00:04<00:00, 116.22it/s]


                 Experiment  Precision@10  Recall@10  NDCG@10
     Exp 1: Standard Cosine        0.0024      0.024   0.0117
Exp 2: User-Centered Cosine        0.0004      0.004   0.0016
Exp 3: Top-50 Pruned Cosine        0.0020      0.020   0.0102


In [8]:
# [Cell 7]: Automatically Pick and Save the Winning Model
# Finding model with highest NDCG@10 (ranking quality)
best_exp_idx = memory_experiments_df["NDCG@10"].idxmax()
best_exp_name = memory_experiments_df.loc[best_exp_idx, "Experiment"]
print(f"\nWinning Model Selected: {best_exp_name}")

matrices_dict = {
    "Exp 1: Standard Cosine": cosine_sim_matrix,
    "Exp 2: User-Centered Cosine": pearson_sim_matrix,
    "Exp 3: Top-50 Pruned Cosine": pruned_cosine_matrix
}

best_matrix = matrices_dict[best_exp_name]

# Save best model to disk
save_npz(MEM_DIR / "best_memory_model.npz", best_matrix)

# Save best metrics for central dashboard
best_metric_row = pd.DataFrame([{
    "Model": f"Memory-Based ({best_exp_name})",
    "Precision@10": memory_experiments_df.loc[best_exp_idx, "Precision@10"],
    "Recall@10": memory_experiments_df.loc[best_exp_idx, "Recall@10"],
    "NDCG@10": memory_experiments_df.loc[best_exp_idx, "NDCG@10"]
}])
best_metric_row.to_csv(MEM_DIR / "memory_metrics.csv", index=False)

# Save lookup artifacts
with open(MEM_DIR / "memory_artifacts.pkl", "wb") as f:
    pickle.dump({
        "best_model_name": best_exp_name,
        "user_to_idx": dict(zip(train_df["user_id"], train_df["user_idx"])),
        "idx_to_game": dict(zip(train_df["game_idx"], train_df["parent_asin"]))
    }, f)

print(f"Successfully saved BEST model to: {MEM_DIR / 'best_memory_model.npz'}")
print(f"Saved metadata to: {MEM_DIR / 'memory_artifacts.pkl'}")


Winning Model Selected: Exp 1: Standard Cosine
Successfully saved BEST model to: best_memory_model.npz
Saved metadata to: memory_artifacts.pkl
